# KAF-5 — Delivery semantics (at-least-once vs exactly-once)

**Break → Detect → Fix → Prove.** Three guarantees, decided by *where you commit* and *whether
writes are idempotent*: **at-most-once** (commit before processing → may **lose**),
**at-least-once** (process then commit / retry → may **duplicate**), and **exactly-once**
(idempotent producer + transactions + `read_committed` consumer + idempotent sink). We make
duplicates *appear*, then walk the machinery that removes them.

**Pre-requisite:** `make up` (Kafka at `localhost:29092` for producers, `kafka:9092` for Spark) —
inspect the topic live in **kafka-ui** at http://localhost:8080. **Laptop-safe:** a bounded
~1,000-message batch, no infinite loops; the topic is deleted at teardown.

> **Honesty note (same stance as the SPK-2/3 OOM modules).** What is *deterministic* here —
> at-least-once duplicates, the idempotent-producer config, and a `read_committed` consumer reading
> committed data — we **run**. Forcing a live producer retry or a transaction abort inside
> `nbconvert` is brittle, so those we **describe precisely with correct snippets**. The real
> exactly-once path in *this* stack is Spark + checkpoint + idempotent Iceberg sink (STR-2).

In [ ]:
from common.kafka_helpers import (
    ensure_topic, produce_events, topic_end_offsets, delete_topic, BOOTSTRAP, SPARK_BOOTSTRAP,
)
from kafka import KafkaProducer, KafkaConsumer
import json
from collections import Counter

TOPIC = "kaf5_payments"
N = 1000            # distinct logical charge events (ids 0..999)
GROUP = "kaf5-app"
print("producers/consumers ->", BOOTSTRAP, "| Spark reads via", SPARK_BOOTSTRAP)
print("kafka-ui:", "http://localhost:8080")

## Step 0 — a 1-partition topic + a "read all and count ids" helper

One partition keeps the demo deterministic (ordering and offsets are simplest to reason about).
The helper drains the topic from the beginning and tallies how many times each `id` appears.

In [ ]:
ensure_topic(TOPIC, num_partitions=1, recreate=True)

def consume_id_counts(topic, isolation_level="read_uncommitted", group=None):
    """Read the whole topic once and return Counter{id: times_seen}. `isolation_level`
    is 'read_uncommitted' (default — sees everything) or 'read_committed' (skips
    aborted/open-transaction records)."""
    c = KafkaConsumer(
        topic,
        bootstrap_servers=BOOTSTRAP,
        group_id=group,
        auto_offset_reset="earliest",
        enable_auto_commit=False,
        isolation_level=isolation_level,
        consumer_timeout_ms=4000,          # stop when no more records arrive (bounded)
        value_deserializer=lambda b: json.loads(b.decode("utf-8")),
    )
    counts = Counter()
    try:
        for msg in c:
            counts[msg.value["id"]] += 1
    finally:
        c.close()
    return counts

print(f"created {TOPIC} (1 partition); helper ready")

## 1. Break it — at-least-once produces duplicates

A non-idempotent producer that "retries" by re-sending the *same logical events* is exactly what
at-least-once looks like on the wire. We write ids `0..999` once, then re-send ids `0..199` (the
"retry after a false-failure").

In [ ]:
# First (legitimate) send: one event per charge, ids 0..999.
produce_events(TOPIC, N, value_fn=lambda i: {"id": i, "amount": 1.0})

# A non-idempotent producer "retries" a slice — the same logical events, sent again.
naive = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    enable_idempotence=False,                      # <- no producer-side dedupe
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)
RESENT = 200
for i in range(RESENT):
    naive.send(TOPIC, value={"id": i, "amount": 1.0})   # duplicate of an already-sent charge
naive.flush(); naive.close()

offsets = topic_end_offsets(TOPIC)
total = sum(offsets.values())
print("offsets on topic :", offsets, "->", total, "records")
print("distinct ids sent:", N, "| deliberately re-sent:", RESENT)

## 2. Detect it

The signature of at-least-once is **offsets produced > distinct logical ids** — the log contains
replays. We confirm it by value: drain the topic and find the ids seen more than once.

In [ ]:
counts = consume_id_counts(TOPIC)                       # read_uncommitted: sees all records
dupes = {i: c for i, c in counts.items() if c > 1}
naive_revenue = sum(counts.values()) * 1.0              # one consumer naively sums every record

print(f"records on topic : {sum(counts.values())}")
print(f"distinct ids     : {len(counts)}")
print(f"duplicate ids    : {len(dupes)}  (e.g. {sorted(list(dupes))[:5]} each seen {next(iter(dupes.values()))}x)")
print(f"naive revenue    : {naive_revenue:.0f}   <- OVERSTATED (true = {N})")

## 3. Diagnose — the three guarantees

The guarantee is the **combination** of commit-order and sink-idempotency, not one flag:

| Guarantee | How it arises | Failure mode | Needs |
|-----------|---------------|--------------|-------|
| **At-most-once** | commit offset **before** processing | crash after commit → message **lost** | nothing (unsafe auto-commit default) |
| **At-least-once** | process **then** commit; producer retries | a retry / redelivered offset → **duplicate** | manual commit after work (KAF-2) + downstream dedupe |
| **Exactly-once (EOS)** | idempotent producer **+** transactions **+** `read_committed` **+** idempotent sink | none (replays absorbed/aborted) | all four together |

**Exactly-once is not "deliver once."** Records may still be delivered more than once at the
transport layer; EOS means each one *affects the result* once, because dedupe + transactional
atomicity make replays harmless.

## 4. Fix it (A) — idempotent producer (dedupes producer *retries*)

`enable_idempotence=True` (+ `acks="all"`) stamps every batch with a **producer id + monotonic
sequence number** per partition. If a *transport retry* re-sends a batch the broker already wrote,
the broker recognises the sequence and **drops the duplicate**. It does **not** dedupe two separate
`send()` calls for the same business event (that's the app's job — Fix D). We configure it below;
forcing an internal retry deterministically under `nbconvert` is unreliable, so the mechanism is
described, not timing-raced.

In [ ]:
# Producer-retry dedupe: a transport-level retry of an already-written batch is dropped by the broker.
idem = KafkaProducer(
    bootstrap_servers=BOOTSTRAP,
    acks="all",                 # all in-sync replicas must ack
    enable_idempotence=True,    # producer-id + per-partition sequence numbers
    max_block_ms=10000,         # never hang the kernel: fail fast if the txn coordinator is unavailable
    value_serializer=lambda v: json.dumps(v).encode("utf-8"),
)
# One bounded send forces the producer-id handshake to complete (an idempotent producer
# that never sends can block on close()). Use a throwaway topic so the demo topic stays clean.
idem.send("kaf5_idem_probe", value={"ok": 1}).get(timeout=10)
idem.close()
print("idempotent producer configured: acks=all, enable_idempotence=True")
print("=> a retried *batch* is deduped by (producer-id, sequence); two distinct send() calls are NOT.")

## 4. Fix it (B) — transactions: atomic all-or-nothing writes  *(described)*

A transactional producer makes a group of records visible **together or not at all**; aborted
records stay in the log but are marked aborted. This is what makes "consume → transform → produce"
one atomic step. The correct shape (kafka-python 2.x exposes these methods, but a live
commit/abort race is brittle inside a notebook, so we show it rather than run it):

```python
p = KafkaProducer(bootstrap_servers=BOOTSTRAP, acks="all",
                  enable_idempotence=True, transactional_id="kaf5-eos-1")
p.init_transactions()
try:
    p.begin_transaction()
    for rec in transformed_batch:
        p.send(OUT_TOPIC, value=rec)
    # tie the *consumed* offsets into the SAME transaction (read-process-write EOS):
    p.send_offsets_to_transaction(consumed_offsets, GROUP)
    p.commit_transaction()      # records + offsets become visible atomically
except Exception:
    p.abort_transaction()       # nothing becomes visible; safe to retry the whole batch
```

The atomic commit of *records + offsets together* is the core of Kafka read-process-write EOS.

## 4. Fix it (C) — `read_committed` consumer (never sees aborted/uncommitted records)

A `read_committed` consumer filters out records from open or aborted transactions; the default
`read_uncommitted` sees everything. Our records were written **non**-transactionally (so they are
"committed" plain writes), which lets us show that `read_committed` returns the same committed data
— the isolation level is real and observable even without driving a live abort here.

In [ ]:
committed = consume_id_counts(TOPIC, isolation_level="read_committed")
uncommitted = consume_id_counts(TOPIC, isolation_level="read_uncommitted")
print(f"read_committed   : {sum(committed.values())} records, {len(committed)} distinct ids")
print(f"read_uncommitted : {sum(uncommitted.values())} records, {len(uncommitted)} distinct ids")
print("Equal here because these were plain (committed) writes; with a real ABORTED transaction,")
print("read_committed would EXCLUDE the aborted records while read_uncommitted would include them.")

## 4. Fix it (D) — idempotent sink (the practical EOS in *this* stack)

Even with (A)–(C), the robust pattern is to make the **write** idempotent: dedupe on a stable key
(`id` / `(partition, offset)` / LSN) so any surviving replay overwrites instead of double-counting.
We simulate the dedupe a keyed Iceberg `MERGE` would do — collapse by `id`, then re-sum revenue.

In [ ]:
# Idempotent read/sink: keep one record per id (what `dropDuplicates(["id"])` or a keyed MERGE does).
deduped_ids = set(counts.keys())                 # collapse replays by stable key
deduped_revenue = len(deduped_ids) * 1.0         # each charge counted once

print(f"naive revenue   (count every record) : {naive_revenue:.0f}   <- overstated")
print(f"deduped revenue (one per id, EOS sink): {deduped_revenue:.0f}   <- correct (= {N})")
print()
print("For Kafka -> Spark -> Iceberg you don't hand-roll this: Structured Streaming's checkpoint")
print("+ an idempotent MERGE/upsert into Iceberg gives effectively-once into the lakehouse (STR-2).")

## 5. Prove it

Same bytes on the wire; the idempotent read recovers the correct distinct count and revenue —
exactly as an idempotent Iceberg `MERGE` would. The topic still physically holds the replays; the
*result* is correct because the sink is idempotent.

In [ ]:
rows = [
    ("at-least-once (naive resend)", sum(counts.values()), len(counts), len(dupes), f"{naive_revenue:.0f}  (overstated)"),
    ("deduped sink (id MERGE)",       sum(counts.values()), len(deduped_ids), 0,     f"{deduped_revenue:.0f}  (correct)"),
]
hdr = ("state", "records", "distinct ids", "dup ids", "revenue")
print(f"{hdr[0]:<30}{hdr[1]:>9}{hdr[2]:>14}{hdr[3]:>9}   {hdr[4]}")
print("-" * 84)
for r in rows:
    print(f"{r[0]:<30}{r[1]:>9}{r[2]:>14}{r[3]:>9}   {r[4]}")
print("\nDuplicates on the topic are harmless once the sink dedupes by a stable key.")

## Takeaways & "in real production…"

- **The guarantee is the *combination*,** not a flag: commit order decides loss vs duplicate;
  idempotency decides whether duplicates matter.
- **Prefer at-least-once + an idempotent sink** over chasing perfect once-only transport — simpler,
  cheaper, survives more failure modes. Duplicates you can dedupe; lost data you can't.
- **Full Kafka EOS** = idempotent producer **+** transactions **+** `read_committed` **+** idempotent
  sink — all four. Use it only when the sink genuinely can't be made idempotent.
- **For Kafka → Spark → Iceberg, don't hand-roll transactions** — lean on the streaming
  **checkpoint + idempotent MERGE** (STR-2) for effectively-once into the lakehouse.
- **Alert on `offsets ≫ distinct keys`** at a sink's input as a duplicate-rate signal, the way you
  alert on consumer lag (KAF-2).

## Teardown

In [ ]:
delete_topic(TOPIC)
delete_topic("kaf5_idem_probe")   # scratch topic from the idempotent-producer demo
print("Deleted", TOPIC, "(`make clean` also clears any local Kafka/.tmp state).")